# 10 · Summary statistics from skewers

The summary statistics of the flux, built bottom-up on toy boxes whose statistics are known. The toy fields and their
statistics are introduced and validated in notebook 08; here they are only generated.

**Overview.**

- **Part A — skewers:** a regular lattice of skewers along $y$ through every box, step by step (`SkewerBundle`,
  cells, pieces, $\tau$); $\tau$ of the lines of `LINE_IDS` ($\mathrm{Ly}\alpha$, Si III 1206, Si II 1193, Si II 1190), periodic.
- **Part B — transmission and overflux:** the normalisation and one skewer from the gas to the overflux.
- **Part C — the same statistics of the density** on the same skewers as the overfluxes of Part D, figure by figure,
  against their exact expectation.
- **Part D — summary statistics of the overfluxes:** the bins and the statistics (D1); in 3D on the lattice of skewers,
  the $(k_\perp, k_\parallel)$ and $(r_\perp, r_\parallel)$ planes, $P$ and $r^2\xi$, the wedges of $\mu$ (D2, D3); in 1D,
  P1D next to a measurement (D4).
- **Part E — the decomposition by order:** P1D and $\xi_{\rm 1D}$ of the total split into the pairs of subset terms,
  order by order (E1) and term by term (E2).
- **Part F — the wedges by order:** $P(k, \mu)$ and $\xi(r, \mu)$ of the total in the wedges of D3, split the same way,
  order by order (F1) and term by term (F2).

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import time, warnings
import numpy as np, jax, jax.numpy as jnp
# float32, the package's precision; skewerjax.config.set_precision("float64") for float64 (docs/precision.md)
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()
use_style()
from skewerjax.utils.plotting import add_panel_letters
from skewerjax.core import LineBundle, phys

In [ ]:
from skewerjax.simulations import ToyBoxes
L = 302.6                          # the box [cMpc]
METAL_TO_HI = {"SiIII": 1 / 2}    # "SiII": 1 / 20; n_X = r_X n_HI: Si III as before; Si II an illustrative toy choice, half of Si III
boxes = ToyBoxes(box_cMpc=L, n_s = -0.4, n_cells=512, n_real=1, base_seed=137, bao_ratio=0.0008, sigma_bao=1.0, r_bao=147.0, metal_to_hi=METAL_TO_HI, velocity="linear", z = 2.0)
Z = boxes.z                        # the redshift of the boxes (at y = 0)
dx = boxes.dx                      # the cell [cMpc]
R_BAO = boxes.r_bao                # the BAO scale [cMpc]
print(f"{boxes.n_real} toy boxes: z = {Z:g}, box L = {L:g} cMpc, {boxes.n_cells}^3 cells of dx = {dx:.4f} cMpc; r_BAO = {R_BAO:g} cMpc")

In [ ]:
# The skewers and their statistics: the same names as in notebook 11
REALISATIONS = None                # the ensemble: the boxes (None: every box)
LINE_IDS = ("HI-LyA", "SiIII-1206")   # "SiII-1193", "SiII-1190"
LINES = LineBundle(list(LINE_IDS), ref_line_id=LINE_IDS[0])   # line 1, the reference: Lyα
N_SK = 64                          # skewers per transverse axis: N_SK^2 per box, L / N_SK apart (N_SK must divide n_cells)
SPACING_KMS = 1.0                  # a point of the paths every <= SPACING_KMS km/s of Hubble flow.
target_dv = 1.0                    # the target cell size for the skewers [cMpc]; determines kmax and rmin (from the parallel direction)
N_PIX = int(np.round(L/target_dv)) # velocity pixels over one period
BATCH = 256                        # skewers per call (memory only)
DETUNING = "linear"                # first-order mapping of v - v_c to a frequency offset
HUBBLE_FLOW = "linear"             # "exact": c ln(a0/a(y)); "linear": (aH)_ref y
A_EFF = 1 / (1 + Z)                # one comoving-to-physical density conversion, at the box redshift (z at y = 0); None: a(y)^-2 per piece
MEAN = "redshift"                  # T_bar(z): the table of the modes in Part B

In [ ]:
MU_EDGES_XI = (0.0, 0.5, 0.8, 0.95, 1.0)   # the wedges of xi in mu = r_par / r (those of DESI DR2 Lya BAO, MU_WEDGES)
MU_EDGES_P = np.sin(np.linspace(0.0, np.pi / 2, 17))   # the 16 wedges of P in mu = k_par / |k|, evenly spaced in the angle to the line of sight: a separate input
idx = np.arange(boxes.n_real) if REALISATIONS is None else np.asarray(REALISATIONS)
ENSEMBLE = f"{idx.size} independent boxes"
EACH = "each box ({n})"            # the legend of the faint curves ({n}: their number)

## Part A — Skewers

### A1 · Skewers through each box

**Layout.** In every box, skewers run along $y$ through the centres of the cell columns (the nodes $x = i_x\,dx$,
$z = i_z\,dx$ of `grf_cells`), from $y = 0$ to $y = L$, so their first and last points are the same place of the periodic
box.

**The set-up** (the parameters cell, with the names of notebook 11), with $p$ the points per cell and
$N_{\rm pix}$ = `N_PIX`:

| Argument | What it controls | Result or cost |
|---|---|---|
| `N_SK` | the skewers per transverse axis, $N_{\rm SK}^2$ per box, $L/N_{\rm SK}$ apart: every $s$-th cell column in $x$ and in $z$, $s = N/N_{\rm SK}$ (`STRIDE`; $N_{\rm SK}$ must divide $N$), from column 0, exactly through the cell centres $x = i\,s\,dx$, $z = j\,s\,dx$ | the number of skewers and the transverse sampling; the cost is linear in the number of skewers |
| `SPACING_KMS` | the spacing of the points of the path along $y$, in km/s of Hubble flow, as in 11: on the toy grid a whole number of points per cell, the fewest with a spacing of at most `SPACING_KMS`, $p = \lceil u\,dx/$`SPACING_KMS`$\rceil$ (`POINTS_PER_CELL`), $N p + 1$ per skewer: every cell is cut into pieces of width $W = dx/p$ | the result until it converges (accurate while $W \ll b$); the cost of $\tau$ is linear in $p$ |
| `N_PIX` | $N_{\rm pix}$ velocity pixels over one period, of width $dv = v_{\rm per}/N_{\rm pix}$: the velocity resolution | the result: the $k$ range of P1D, up to $\pi/dv$, and the LoS resolution of Part D; the cost of $\tau$ is linear in $N_{\rm pix}$ |

Here $v_{\rm per}$ is the velocity span of one box along a skewer, and $u = v_{\rm per}/L$ converts comoving distances to
velocities.


- **The drift along $v$.** The skewers are in the exact frame: $z$ sits at $y = 0$ and $a(y)$ decreases along the box,
  by $\Delta\ln a = v_{\rm per}/c$, while the box is wrapped as periodic. A uniform gas then has
  $\tau \propto a^{-2}/(aH)$ per pixel, varying across the box, so $\bar T(v)$ drifts and
  jumps back at the wrap. It comes from the factor $a(y)^{-2}$ and from the Jacobian $1/(aH(y))$: `A_EFF` removes the
  first, `HUBBLE_FLOW = "linear"` the second. `"redshift"` removes that drift from $\delta$; with `"global"` it stays as a pattern shared by
  every skewer, which shows as horizontal bands in the $(r_\perp, r_\parallel)$ map (D2).
- **The density conversion, `A_EFF = 1/(1+z)`.** `compute_optical_depth(a_eff=)` replaces $a_p^{-2}$ of every piece by
  one $a_{\rm eff}^{-2}$: one fixed comoving-to-physical conversion at the box redshift ($z$ at $y = 0$; `None` keeps
  $a(y)^{-2}$ per piece). It touches nothing else. With `HUBBLE_FLOW = "exact"` the line centres keep $c\ln(a_0/a(y))$, so a column still spreads
  over a velocity width $\propto aH\,\ell$: the Jacobian $1/(aH)$ of the frame remains, about $-1.8$ % across the box
  ($aH$ grows with $z$).
- **The detuning, `DETUNING = "linear"`.** fake_spectra's first-order mapping of $v - v_c$ to a frequency offset in
  the line profile. It changes $\tau$ by less than $10^{-5}$, and it does not linearise the line centres
  $c\ln(a_0/a(y))$ of the exact frame.
- **The Hubble flow, `HUBBLE_FLOW = "exact"`.** A piece at path coordinate $y_p$ absorbs at line centre
  $v_{c,p} = V_H(y_p) + v_{{\rm pec},p}$. `"exact"` follows the frame, $V_H(y) = c\ln(a_0/a(y)) = \int_0^y aH\,dy'$:
  the line centres are spaced by $aH(y)$, which grows along the box, so a column spreads over a velocity width that
  varies (the Jacobian $1/(aH)$), and $v_{\rm per} = c\ln(a(0)/a(L))$. `"linear"` is fake_spectra's Hubble flow,
  $V_H(y) = (aH)_{\rm ref}\,y$ with $(aH)_{\rm ref}$ at $z$ ($y = 0$): uniform spacing, a constant Jacobian and
  $v_{\rm per} = (aH)_{\rm ref}L$. The two differ at second order, $V_H - (aH)_{\rm ref}y \simeq \tfrac12 (aH)'\,y^2$,
  108 km/s over one box. Together with `A_EFF = 1/(1+z)` and `DETUNING = "linear"`, `"linear"` is fake_spectra's
  model of one snapshot; what remains is numerical (fake_spectra averages each profile over the pixel and over the
  cell's segment, while here it is evaluated at the pixel centre and each piece at its centroid).

In [ ]:
from tqdm.auto import tqdm
from skewerjax.core import SkewerBundle, bundle_frame, compute_optical_depth, lattice_columns, lattice_points, velocity_span
from skewerjax.core.absorbers import cell_pieces
from skewerjax.summary_stats import lattice_limits

In [ ]:
STRIDE = (boxes.n_cells // N_SK,) * 2                             # every STRIDE-th cell column in x and in z
assert N_SK * STRIDE[0] == boxes.n_cells, f"N_SK = {N_SK} must divide n_cells = {boxes.n_cells}"
COLS = lattice_columns(boxes.n_cells, STRIDE)                     # (N_SK^2, 2) the cell columns (i_x, i_z) of the skewers, x-major
D_PERP = L / N_SK                                                 # the transverse spacing [cMpc]
FRAME_KW = dict(cosmologies=boxes.cosmology(), a0=1 / (1 + Z), x0=L, ICs_defined_at_index=-1)  # the exact frame: z at y = 0, the first point


def paths(cols, p):
    # the SkewerBundle of the skewers through the cell columns cols, p points per cell along y: paths and frame
    return SkewerBundle(spatial_points=lattice_points(cols, dx, L, boxes.n_cells, p), **FRAME_KW)


u0 = float(velocity_span(paths(COLS[:1], 1), HUBBLE_FLOW)[0]) / L # km/s per cMpc, to choose p (the span does not depend on the sampling)
POINTS_PER_CELL = int(np.ceil(u0 * dx / SPACING_KMS))             # p: the fewest points per cell with a spacing <= SPACING_KMS
SK0 = paths(COLS[:1], POINTS_PER_CELL)                            # one skewer: the geometry
FRAME = bundle_frame(SK0)                                         # the exact frame: a0 = 1 / (1 + z) at y = 0, a(y) along the path
PERIOD = float(velocity_span(SK0, HUBBLE_FLOW)[0])                # v_per [km/s], one box: c ln(a(0)/a(L)) ("exact") or (aH)_ref L ("linear"); not DETUNING
N_LOS = boxes.n_cells * POINTS_PER_CELL                           # points along a skewer, the end point (= the first) dropped
DS = L / N_LOS                                                    # their spacing [cMpc]: dx / p
u = PERIOD / L                                                    # km/s per cMpc: v = u x
V = (np.arange(N_PIX) + 0.5) * PERIOD / N_PIX                     # the velocity grid: N_PIX pixel centres over the period (Lyα axis)
DV = PERIOD / N_PIX
lim = lattice_limits(L, D_PERP, N_PIX, PERIOD)                    # the lattice of Part D: exactly the skewers' samples
print(f"skewers: {N_SK} x {N_SK} = {len(COLS)} per box (every {STRIDE[0]}-th cell column), {D_PERP:.4f} cMpc apart; "
      f"{len(COLS) * idx.size} in all ({ENSEMBLE}), in batches of {BATCH}")
print(f"points: {N_LOS} per skewer, {POINTS_PER_CELL} per cell, every {DS * 1e3:.2f} ckpc = {DS * u:.3f} km/s (the pieces: W = dx / p); "
      f"v_per = {PERIOD:.2f} km/s; pixels: {N_PIX} of dv = {DV:.2f} km/s, P1D up to pi / dv = {np.pi / DV:.4f} s/km")
print(f"SkewerBundle: frame a0 = {float(FRAME.a_dagger):.5f} = 1 / (1 + z) at y = 0 (z = {Z:g}), a(L) = {float(np.asarray(SK0.scale_factors)[0, 0, -1]):.5f}")

**$\tau$, box by box.** Each box is generated from its seed and `compute_optical_depth` sums them into $\tau$.
Only $\tau$ and the densities of the cells the skewers cross are kept.

In [ ]:
TAU, DENS, RES, TIMES = [], [], [], []   # tau (N_lines, N_sk, N_PIX) per box, the gas the lattice crosses, resolution checks, timings
t0 = time.perf_counter()
bar = tqdm(total=len(idx), desc="boxes")
with warnings.catch_warnings(record=True) as WARN:               # the resolution check of compute_optical_depth (counted below)
    warnings.simplefilter("always")
    for n_done, i in enumerate(idx, start=1):
        t_f = time.perf_counter()
        fields_i = boxes.fields(int(i))                          # the gas of realisation i on the grid
        cells = boxes.cells(int(i), fields_i)                    # -> core.Cells
        t_f, t_p, t_t, tau_i = time.perf_counter() - t_f, 0.0, 0.0, []
        for b in range(0, len(COLS), BATCH):
            t1 = time.perf_counter()
            cols_b = COLS[b:b + BATCH]
            pieces = cell_pieces(cells, LINES, skewer_bundle=paths(cols_b, POINTS_PER_CELL))  # cells -> pieces
            t2 = time.perf_counter()
            tau_b, info = compute_optical_depth(  # CALCULATION OF THE OPTICAL DEPTH tau
                pieces, V, LINES, line_frame="reference", velocity_period=True, detuning=DETUNING,
                hubble_flow=HUBBLE_FLOW, aH_ref=None, a_eff=A_EFF, resolution_tolerance=0.25, return_info=True
            )
            tau_i.append(np.asarray(tau_b))
            RES.append(np.stack([info["resolution"]["max_ratio"], info["resolution"]["column_fraction"]], -1))
            t_p, t_t = t_p + t2 - t1, t_t + time.perf_counter() - t2
        TAU.append(np.concatenate(tau_i, axis=1))
        DENS.append({sp: n[::STRIDE[0], :, ::STRIDE[1]] for sp, n in fields_i["n"].items()})  # the cells the lattice crosses
        TIMES.append((t_f, t_p, t_t))
        del fields_i, cells
        elapsed = time.perf_counter() - t0
        bar.set_postfix_str(f"fields {t_f:.1f} s · pieces {t_p:.1f} s · τ {t_t:.0f} s · elapsed {elapsed:.0f} s · "
                            f"ETA {elapsed / n_done * (len(idx) - n_done):.0f} s")
        bar.update()
bar.close()
TAU, TIMES, RES = np.stack(TAU), np.array(TIMES), np.array(RES)  # TAU (n_ens, N_lines, N_sk, N_PIX)
t_skewers = time.perf_counter() - t0
del pieces

In [ ]:
from skewerjax.utils.plotting import line_names
NAMES_L = line_names(LINE_IDS)                                   # Lyα, Si III, Si II 1193, Si II 1190
n_ens, n_sk = TAU.shape[0], TAU.shape[2]                         # the members of the ensemble (boxes) and the skewers of each
T_REF = boxes.T0                                                 # the temperature of the thermal widths b [K]: the toy's T0
b_th = {lid: np.sqrt(2 * phys.kB * T_REF / m) / 1e3 for lid, m in zip(LINE_IDS, np.asarray(LINES.mass_amu))}
print(f"{n_ens} boxes x {n_sk} skewers in {t_skewers:.0f} s; per box: fields {TIMES[:, 0].mean():.1f} s, pieces {TIMES[:, 1].mean():.1f} s, "
      f"tau {TIMES[:, 2].mean():.0f} s")
print(f"resolution check (W <= b/4): max W / b = {np.round(RES[..., 0].max(axis=0), 3)}, column fraction of the pieces wider than b/4 "
      f"at most {np.round(RES[..., 1].max(axis=0), 6)} ({', '.join(NAMES_L)}); warnings: {len({str(w.message) for w in WARN})} distinct, "
      f"{len(WARN)} in all; b at {T_REF:g} K: " + ", ".join(f"{lid} {b:.2f} km/s" for lid, b in b_th.items()))
print(f"axis: a cell spans {PERIOD / boxes.n_cells:.1f} km/s ({N_PIX / boxes.n_cells:g} pixels), a piece {PERIOD / boxes.n_cells / POINTS_PER_CELL:.2f} km/s")

## Part B — Transmission and overflux

**Transmission and overflux.** $T = e^{-\tau}$ is the **transmission**; the flux is the observed $F = F_0\,T$, the
continuum $F_0$ times the transmission. The overflux follows continuum-fitting convention:

$$
\delta = \frac{F}{F_0\,\bar T} - 1 = \frac{T}{\bar T} - 1,\qquad \bar T = \langle T\rangle,
$$

the mean over a set of pixels chosen by `MEAN`. Each line $j$ has $\delta_j = T_j/\bar T_j - 1$, and the
total, $T = \prod_j T_j$, has $\bar T = \langle\prod_j T_j\rangle \neq \prod_j \bar T_j$.

**The normalisation modes.** $\langle\delta\rangle = 0$ holds over the set of pixels of the mean:

| `mean` | $\bar T$ over | removes | use it for |
|---|---|---|---|
| `"redshift"` (default) | the pixels at the same redshift, of every skewer of every box: $\bar T(z)$ | the mean-flux evolution along the skewers, a pattern shared by all of them | the observational convention, the mean flux as a function of $z$ |
| `"global"` | every pixel of every skewer of every box: one number | only the overall mean; a drift of $\bar T$ along $v$ stays in $\delta$ | a path without evolution |
| `"box"` | every pixel of every skewer of one box | the box-to-box scatter of the mean ($\mathbf k = 0$ of each box) | normalising each realisation by itself |
| `"skewer"` | the pixels of one skewer | the $k_\parallel = 0$ mode of every skewer | mimicking a fitted continuum amplitude |

All skewers share one grid of pixel scale factors here, so $\bar T(z)$ is the mean at each pixel over the ensemble.
**Here `MEAN = "redshift"`** (the parameters cell).

**The continuum.** Here $F_0$ is known (in effect $F/F_0 = T$), so $\delta = T/\bar T - 1$ exactly. In data, $F_0$
must be estimated for each quasar. A per-quasar change in the continuum's amplitude or slope is indistinguishable from
a large-scale fluctuation of the transmission along that sightline, so even a correct continuum model absorbs those
modes of $\delta$; analyses correct for this with a projection. It does not arise here, because $F_0$ is known.

In [ ]:
from skewerjax.summary_stats import ensemble_mean_transmissions, overflux_check
A_PIX = FRAME.a_dagger * np.exp(-V / phys.c_kms)                                # the scale factor of every pixel: the redshift grid of the mean
T = np.exp(-TAU)                                                                # the transmission of each line (n_ens, N_lines, N_sk, N_PIX)
T_BAR = ensemble_mean_transmissions(T, mean=MEAN, grid=A_PIX)                   # T_bar of every line and of the total: (n_ens, N_lines + 1, N_sk, N_PIX), a view
DELTA = np.concatenate([T, T.prod(axis=1, keepdims=True)], axis=1) / T_BAR - 1  # delta = T / T_bar - 1 (every line, total)
CHECK = overflux_check(DELTA, T_BAR, mean=MEAN, grid=A_PIX)                     # <delta> = 0 over the sets of MEAN, C, the decomposition
T_v = ensemble_mean_transmissions(T, mean=MEAN)[0, :, 0]                        # T_bar(v) at each pixel over the ensemble, whatever MEAN (N_lines + 1, N_PIX)
drift = T_v / T_v.mean(axis=1, keepdims=True) - 1                               # the drift along v of each line and the total
print("mean transmission over every pixel: " + ", ".join(f"{nm} {t.mean():.4f}" for nm, t in zip([*NAMES_L, "total"], T_v)) + f". scale factor from {A_PIX[0]:.5f} to {A_PIX[-1]:.5f}")
print("drift of T_bar(v) along v, T_bar(v) / <T_bar> - 1 (diagnostic): " + ", ".join(
      f"{n} {d[:5].mean():+.4f} -> {d[-5:].mean():+.4f} (rms {d.std():.1e})" for n, d in zip([*NAMES_L, "total"], drift))
      + f"; Delta ln a = v_per / c = {PERIOD / phys.c_kms:.4f}")
print(f"normalisation: max |<delta>| over the sets of MEAN = {CHECK['mean_delta_level'].max():.1e} (every line, the "
      f"total); C = T_bar / prod_j T_bar_j - 1 = {CHECK['C']:.4e} (mean), |C - sum_(|A| >= 2) <delta_A>| = {CHECK['C_error']:.1e}; "
      f"max |[sum_A delta_A - C] / (1 + C) - delta_tot| = {CHECK['decomposition_error']:.1e}")
del T

### B1 · One skewer, from the gas to the flux contrast

In [ ]:
from skewerjax.utils.plotting import plot_skewer_slice, plot_toy_skewer
i0 = 0                                                                    # the member of the ensemble shown: the first box
on_z0 = np.flatnonzero(COLS[:, 1] == 0)
j = int(on_z0[np.argmin(np.abs(COLS[on_z0, 0] - boxes.n_cells // 2))])   # the column nearest to x = L/2 with i_z = 0
ix, iz = COLS[j]
fields0 = boxes.fields(int(idx[i0]))                                      # its gas on the grid (generated again: the run keeps the lattice only)
fig, ax, sl_info = plot_skewer_slice(fields0, L, COLS, z_index=0, highlight=(ix, iz), cmap="hot")
plt.show()
print(f"{sl_info['n_row']} skewers in the layer i_z = 0, {sl_info['spacing_cMpc']:.4g} cMpc apart; the skewer of the next figure: "
      f"x = {ix * dx:.1f} cMpc")

**Along the skewer.** Rows (a) and (b) are along the comoving position $y$ (the $N$ cell values, as steps); rows (c)
to (e) along the velocity of the $\mathrm{Ly}\alpha$ axis, on which every other line $j$ is evaluated at $v + \Delta v_j$ and wrapped
into the period. The overdensities (b) of H I, Si III and Si II are identical by construction (one gas, with one temperature);
their absorption is not: the metal lines are narrower ($b \propto m^{-1/2}$) and displaced by the separations $\Delta v_{\mathrm{Ly}\alpha\text{–}j}$ of the
lines, wrapped into the period (printed below). Row (c) is the transmission $T = e^{-\tau}$ of each line and the total
$T = \prod_j T_j$; row (d) the mean transmission $\bar T$ that normalises this skewer, from the step (c) of Part B
(`MEAN`: $\bar T(z)$ of the ensemble with `"redshift"`; that skewer's or box's value with `"skewer"` or `"box"`); row (e)
their overfluxes, $\delta = T/\bar T - 1$, so the mean of (e) along this one skewer need not vanish. The very thin grey
vertical lines are the grids: in (a) and (b) the points of the path ($p$ per cell), in (c) to (e) the pixel centres,
where $\tau$ is evaluated. When
the lines of a panel would be too dense to read, they are drawn only in a zoomed inset, whose range is marked on the
panel.

In [ ]:
from skewerjax.utils.plotting import line_pairs
n_skewer = {sp: fields0["n"][sp][ix, :, iz] for sp in boxes.species}     # the density of the cells along this skewer
fig, axs, grid_info = plot_toy_skewer(L, n_skewer, V, TAU[i0][:, j], mean_transmission=T_BAR[i0][:, j], points_per_cell=POINTS_PER_CELL, line_ids=LINE_IDS)
add_panel_letters(axs)
plt.show()
PAIRS = line_pairs(LINE_IDS, np.asarray(LINES.delta_v), PERIOD)           # every pair of lines, its separation folded into [0, v_per / 2]
dv_12 = float(LINES.delta_v[1])
dv_fold = PAIRS[0]["fold"]                                                # Si III - Lya, folded into half the period
print(f"line separations Delta v [km/s], folded into [0, v_per / 2] with v_per = {PERIOD:.0f} km/s [km/s and cMpc]: " + "; ".join(
      f"{NAMES_L[p['i']]}–{NAMES_L[p['j']]} {p['dv']:.0f} -> {p['fold']:.0f} = {p['fold'] / u:.1f}" for p in PAIRS))
print(f"b(Lya) / b(Si) = {b_th[LINE_IDS[0]] / b_th[LINE_IDS[1]]:.2f}")

## Part C — statistics of the density on the skewers

The skewer run samples the gas along every skewer, one-to-one with the overflux skewers of Part D. Part C puts the
overdensity on those skewers through exactly the figures of Part D (the same functions, options, conventions, etc.),
so each figure of C can be compared with its counterpart in D.

- **The fields.** The overdensities of $n_{\rm HI}$, of $n_{\rm SiIII}$ and of their sum, against its global mean
  over every point of every skewer of every box (the density has no evolution along $y$),
  $\delta_X = n_X/\langle n_X\rangle - 1$. In this toy model every metal species is a fixed ratio of $n_{\rm HI}$, $n_X = r_X n_{\rm HI}$ (`metal_to_hi`).
- **The LoS is a comoving distance**, in cMpc, sampled at the points of the skewers, spacing
  $dx/p$ ($p$ = `points_per_cell`), the duplicated periodic end point dropped, so the lattice is exactly periodic.
- **The expectation.** the density has an exact expectation on this lattice: $\xi_{\rm grid}$ at its
  separations (linear between the cells along the LoS) and the transversely aliased spectrum
  (`lattice_density_theory`), drawn dashed.

**The bins of Part C** (next cell), passed to the 3D statistics of the density (`lattice_grid`), from the lattice of
the points of the skewers, in cMpc: $s\,dx$ across, $dx/p$ along the line of sight. They are independent of the bins of
the overfluxes (D1).

- $r_\perp$, `RPERP_EDGES_B` (`set_rperp_edges`): the first bin holds $r_\perp = 0$ alone, the pairs on the same skewer;
  then one bin per distinct skewer separation $\sqrt{i^2 + j^2}\,s\,dx$ up to $r_1$, then steps of the skewer spacing to $L/2$.
- $r_\parallel$: the lattice's own separations, one row per point separation $j\,dx/p$ from 0 to $L/2$, each row
  centred on its separation.
- $\lvert\mathbf r\rvert$, `R_EDGES_B`, also the $\lvert\mathbf r\rvert$ bins of the wedges (`set_r_edges`, the
  counterpart of `set_k_edges`): one bin per distinct separation up to $r_1$ (here the first two point separations along
  the line of sight, $dx/p$ and $2\,dx/p$: $r_1 = 3\,dx/p$ rounds down to the edge halfway between $2\,dx/p$ and $3\,dx/p$), then steps of the coarser spacing $\max(s\,dx, dx/p)$ to $L/2$. The zero lag
  (the variance) belongs to no bin: the first edge is half the smallest separation. Every bin holds separations (the
  occupancy is printed).
- $\lvert\mathbf k\rvert$, `K_EDGES_B`: one bin per mode, then steps of $0.4\,k_f$ to $3/\sigma_{\rm BAO}$, then logarithmic (`set_k_edges`), up to the smaller Nyquist frequency of this lattice,
  $\min\bigl(\pi/(s\,dx), \pi p/dx\bigr)$, so that every bin holds all directions.

Everything is computed on every mode and separation of the lattice; the maps are drawn over the range they share with
the overflux maps of D2, $k_\parallel \le \pi u/dv$ and $r_\parallel \le v_{\rm per}/2$ ($L/2$).

In [ ]:
from skewerjax.summary_stats import lattice_grid, set_k_edges, set_r_edges, set_rperp_edges
# The bins of the 3D statistics of the density (Part C), from the lattice of its points [cMpc], [cMpc^-1]
k_nyq_b = min(np.pi / D_PERP, np.pi / DS)                                     # the smaller Nyquist frequency
RPERP_EDGES_B, RPERP_INFO_B = set_rperp_edges(D_PERP, L=L, r_1=3 * D_PERP, r_2=L / 2, r_max=L / 2, dr_factor=1.0, n_log=0, merge_empty=True, info=True)   # r_perp: one bin per skewer separation to r_1 (the first: the same skewer), then steps of d_perp to L / 2
R_EDGES_B, R_INFO_B = set_r_edges(D_PERP, DS, L=L, r_1=3 * DS, r_2=L / 2, r_max=L / 2, dr_factor=0.2, merge_empty=True, info=True)   # |r|: one bin per separation to r_1 = 3 ds, then steps of max(d_perp, ds) to L / 2
K_EDGES_B = set_k_edges(2 * np.pi / L, k_1=3.5 * 2 * np.pi / L, k_2=k_nyq_b, k_max=k_nyq_b, dk_factor=0.4)   # |k|: one bin per mode to k_1, then steps of 0.4 k_f to k_2 = k_max
EDGES_B = dict(k_edges=K_EDGES_B, r_edges=R_EDGES_B, rperp_edges=RPERP_EDGES_B)
print(f"Part C bins: r_perp {RPERP_EDGES_B.size - 1} ({RPERP_INFO_B['n_sep']} one per skewer separation to {RPERP_INFO_B['r_1']:.3f} cMpc, the first "
      f"r_perp = 0; {RPERP_INFO_B['n_step']} steps of {RPERP_INFO_B['dr']:.4g} cMpc to {RPERP_EDGES_B[-1]:.1f} cMpc); r_par one row per point separation, "
      f"{N_LOS // 2 + 1} rows from 0 to L / 2; |r| {R_EDGES_B.size - 1} ({R_INFO_B['n_sep']} one per separation, from {R_EDGES_B[0]:.4f} to "
      f"{R_INFO_B['r_1']:.4f} cMpc; {R_INFO_B['n_step']} steps of {R_INFO_B['dr']:.4g} cMpc to {R_EDGES_B[-1]:.1f} cMpc; separation vectors per bin "
      f"{R_INFO_B['n_lags'].min()} to {R_INFO_B['n_lags'].max()}, {R_INFO_B['n_merged']} empty bin(s) merged); |k| {K_EDGES_B.size - 1} up to {K_EDGES_B[-1]:.4f} cMpc^-1")
print(f"full coverage of the lattice of the points: |k| <= min(pi / d_perp, pi / ds) = min({np.pi / D_PERP:.4f}, {np.pi / DS:.4f}) "
      f"cMpc^-1, |r| <= L / 2 = {L / 2:.1f} cMpc")

In [ ]:
from skewerjax.summary_stats import lattice_density_theory, skewer_density_statistics
GRID_B = lattice_grid(N_SK, N_SK, N_LOS, L, D_PERP, **EDGES_B, xi_mu_edges=MU_EDGES_XI, pk_mu_edges=MU_EDGES_P)
# skewer samples: on the skewers' own lattice, along the line of sight N_LOS = n_cells x POINTS_PER_CELL points (the periodic end point dropped), across it the skewer spacing; no deposition, window or interlacing
DENS_STATS = skewer_density_statistics(DENS, POINTS_PER_CELL, GRID_B, L, n_bar=boxes.n_bar)
# the reference: the exact expectation of the density on this lattice (lattice_density_theory)
REF_B = lattice_density_theory(boxes.grf().power, GRID_B, boxes.n_cells, STRIDE, points_per_cell=POINTS_PER_CELL, scale=DENS_STATS["density_scale"])
axB, dB = DENS_STATS["density_axes3d"], DENS_STATS["density3d"]
SP = boxes.species
NAMES_B = {**{sp: rf"$n_{{\rm {sp}}}$" for sp in SP}, "total": "$" + " + ".join(rf"n_{{\rm {sp}}}" for sp in SP) + "$"}
LIM_B = GRID_B.limits()                                                      # the full coverage of this lattice (coverage_limits): k_f, the smaller Nyquist frequency, the smallest separation ds, L / 2
print(f"lattice: {N_SK} x {N_LOS} x {N_SK} points per box, spacing {DS:.4f} cMpc = dx / p along the line of sight "
      f"({u * DS:.2f} km/s), {D_PERP:.4g} cMpc across")
print("global means <n_X> / n_bar_X: " + ", ".join(f"{f} {DENS_STATS['density_mean'][f]:.6f}" for f in DENS_STATS['density_mean'])
      + f"; identical overdensities (n_X = r_X n_HI): max |delta_HI - delta_X| = {DENS_STATS['density_identity'][0]:.1e}, "
      f"max |delta_HI - delta_total| = {DENS_STATS['density_identity'][1]:.1e}")

### C1 · The $(k_\perp, k_\parallel)$ & $(r_\perp, r_\parallel)$ planes

In [ ]:
from skewerjax.utils.plotting import plot_overflux_kpar_kperp, plot_overflux_rpar_rperp
densE = dict(dB["total"], axes=axB)
fig, ax, chi2_B = plot_overflux_kpar_kperp(
      densE, kms_per_cMpc=u, kperp_edges=K_EDGES_B, n_sigma=1, unit="cMpc", theory=REF_B, kpar_max=k_nyq_b,
      show_mu=True, show_contours=False, show_modes=True,
)
plt.show()

In [ ]:
fig, ax, curves_B = plot_overflux_rpar_rperp(
    densE, kms_per_cMpc=u, rperp_edges=RPERP_EDGES_B, r_bao=R_BAO, n_sigma=1,
    color_scale="symlog", quantity="r2xi", unit="cMpc", theory=REF_B, levels=[5*10**-5],
    show_mu=True, show_contours=False, show_pairs=False,
    )
plt.show()
print(f"map: {axB['rpar_edges'].size - 1} rows, one per separation j L / N_LOS from 0 to L / 2")

### C2 · The isotropic statistics of the density, against the expectation

$P(\lvert\mathbf k\rvert)$ and $\xi(\lvert\mathbf r\rvert)$ of every species and of their sum at the points of the
skewers, in the bins of Part C over the full coverage of this lattice (`LIM_B`), against the exact expectation on this
lattice (dashed; one curve: every species has the same overdensity). The same figure as D3 for the overfluxes.

In [ ]:
from skewerjax.utils.plotting import plot_overflux_3d
fig, axs = plot_overflux_3d(dB, axB, kms_per_cMpc=u, r_bao=R_BAO, names=NAMES_B, k_edges=K_EDGES_B, r_edges=R_EDGES_B, limits=LIM_B, theory=REF_B,
                            each_label=EACH, quantity="r2xi", y_scale="linear", k_scale="linear")
add_panel_letters([axs["P"], axs["xi"]])
plt.show()

**The integral constraint.** The density has an analytic model: as in 08 (A3), the continuous $\xi(r)$ minus the mean
of $\xi$ over the box, $\bar\xi_V$ (`integral_constraint`), against the expectation on this lattice.

In [ ]:
from skewerjax.summary_stats import integral_constraint
GRF = boxes.grf()
xi_cont = lambda r: np.asarray(GRF.correlation(r), dtype=float)                    # the closed form of the toy model
XI_BAR = integral_constraint(xi_cont, L)
r_b = axB["r_iso"]
ok_b = np.isfinite(REF_B["xi_iso"]) & (r_b > 0)
dev_b = np.abs(xi_cont(r_b[ok_b]) - XI_BAR - REF_B["xi_iso"][ok_b])
print(f"xi_bar_V = {XI_BAR:.4e}; max |(xi - xi_bar_V) - expectation on the lattice|: r < L / 4 {dev_b[r_b[ok_b] < L / 4].max():.2e}, r < L / 2 {dev_b.max():.2e}; "
      f"r >= 6 dx and r < L / 4 {dev_b[(r_b[ok_b] >= 6 * dx) & (r_b[ok_b] < L / 4)].max():.2e}, r >= 6 dx and r < L / 2 {dev_b[r_b[ok_b] >= 6 * dx].max():.2e}")

## Part D — Summary statistics of the overfluxes

**Two kinds of axis.** Along the line of sight the coordinate is a **velocity**, $v$ [km/s]; across it, the coordinates
are comoving **distances**, $x_\perp$ [cMpc]. One approximate factor relates them, the mean Hubble rate along a skewer
($dv/dy = aH$ changes along it with $a(y)$):

$$
\Delta v \simeq u\,\Delta x,\qquad u = \frac{1}{L}\int_0^L aH\,dy = \frac{v_{\rm per}}{L}.
$$

Two rules follow:

- **Maps** ($r_\parallel$ against $r_\perp$): each axis keeps its native coordinate. The bottom axis is the
  perpendicular one, in cMpc; the left axis is the line of sight, in km/s.
  For reference only, a top axis gives the perpendicular
  coordinate in km/s and a right axis the line of sight in cMpc, both through $u$.
- **Isotropic quantities that combine both axes** ($\lvert\mathbf r\rvert$, $\lvert\mathbf k\rvert$, $\mu$, $P$, etc.):
  the line of sight is converted to comoving distance, $r_\parallel = v/u$ and $k_\parallel = u\,k_v$, and they are
  reported in comoving units: $r$ in cMpc, $r^2\xi$ in $\mathrm{cMpc}^2$, $k$ in $\mathrm{cMpc}^{-1}$ (the top axes give km/s and s/km for
  reference). On the lattice a pixel $dv$ spans $dv/u = L/N_{\rm pix}$ cMpc, so the lattice is a cube of side $L$.

**What the lattice resolves.** The statistics are computed on exactly the lattice of the skewers, with no regridding.
Along the LoS: from the pixel size $dv$ up to the span of the box, $v_{\rm per}$. Across it:
from the skewer spacing $s\,dx$ (set by `stride`) up to $L$.

**Errors.** Points with error bars are the mean over the boxes and its error.

### D1 · The bins and the statistics

**The statistics, from arrays and geometry only.** P1D and $\xi_{\rm 1D}$ by one 1D FFT per skewer
(`skewer_p1d`, `skewer_xi1d`), the 3D statistics on the lattice of the skewers with the bins `EDGES_C` (`lattice_grid`,
`measure_lattice`), and, in Part C, the density on the same skewers (`skewer_density_statistics`, `lattice_density_theory`): no
box class enters, only $\delta$, $dv$, the lattice spacing and the bins.

**The bins of the overfluxes** (next cell), passed to the 3D statistics (`lattice_grid`) and to the figures of D2 and D3. The statistics use exactly
the lattice the skewers produce, with no regridding or interpolation before the FFT: $s\,dx$ across, $dv$ along the line
of sight (printed above), converted to comoving units with the one factor $u$ for the isotropic quantities (above).

- $r_\perp$, `RPERP_EDGES_C` (`set_rperp_edges`): the first bin holds $r_\perp = 0$ alone, the pairs on the same skewer;
  then one bin per distinct skewer separation up to $r_1 = 3\,s\,dx$, then steps of $s\,dx$ to $L/2$.
- $r_\parallel$: the lattice's own separations, one row per pixel separation, $r_\parallel = j\,dv$ from 0 to
  $v_{\rm per}/2$, each row centred on its separation (no array: they follow from $N_{\rm pix}$).
- $\lvert\mathbf r\rvert$, `R_EDGES_C`, also the $\lvert\mathbf r\rvert$ bins of the wedges (`set_r_edges`): one bin
  per distinct separation of the lattice up to $r_1 = 2\,s\,dx$ (combinations of $j\,dv/u$ along and $s\,dx$ across: few
  at small $r$), then steps of the coarser spacing $\max(s\,dx, dv/u)$ to $L/2$; the zero lag in no bin, so the first bin
  holds $r = dv/u$ alone, the smallest separation and the shaded $r_{\min}$ of D3.
- $\lvert\mathbf k\rvert$, `K_EDGES_C`: the binning of Part C up to the smaller of the two Nyquist frequencies in
  $\mathrm{cMpc}^{-1}$, $\pi/(s\,dx)$ and $\pi u/dv$, so that every bin holds all directions; also the $\lvert\mathbf k\rvert$ bins of the wedges of $P$.
- The wedges: `MU_EDGES_XI` in $\mu = r_\parallel/r$ and `MU_EDGES_P` in $\mu = k_\parallel/\lvert\mathbf k\rvert$
  (configuration), with $r_\parallel$, $k_\parallel$ along the skewers in comoving units (through $u$); the same in Part C.

In [ ]:
# The bins of the 3D statistics of the overfluxes (D1) in comoving units [cMpc], [cMpc^-1] (the line of sight converted with u)
dv_c = DV / u                                                                   # a pixel dv / u [cMpc]
k_nyq_c = min(np.pi / D_PERP, np.pi / dv_c)                                     # the smaller Nyquist frequency
RPERP_EDGES_C, RPERP_INFO_C = set_rperp_edges(D_PERP, L=L, r_1=3 * D_PERP, r_2=L / 2, r_max=L / 2, dr_factor=1.0, n_log=0, merge_empty=True, info=True)   # r_perp: one bin per skewer separation to r_1 (the first: the same skewer), then steps of d_perp to L / 2
R_EDGES_C, R_INFO_C = set_r_edges(D_PERP, dv_c, L=L, r_1=2 * D_PERP, r_2=L / 2, r_max=L / 2, dr_factor=0.8, merge_empty=True, info=True)   # |r|: one bin per separation to r_1, then steps of max(d_perp, dv / u) to L / 2
K_EDGES_C = set_k_edges(k_f = 2 * np.pi / L, k_1 = 3.5 * 2 * np.pi / L, k_2 = k_nyq_c, k_max = k_nyq_c, dk_factor=0.4)   # |k|: as Part C, to the smaller Nyquist frequency
EDGES_C = dict(k_edges=K_EDGES_C, r_edges=R_EDGES_C, rperp_edges=RPERP_EDGES_C)
print(f"overflux bins: r_perp {RPERP_EDGES_C.size - 1} ({RPERP_INFO_C['n_sep']} one per skewer separation to {RPERP_INFO_C['r_1']:.3f} cMpc, the first "
      f"r_perp = 0; {RPERP_INFO_C['n_step']} steps of {RPERP_INFO_C['dr']:.4g} cMpc to {RPERP_EDGES_C[-1]:.1f} cMpc); r_par one row per pixel separation, "
      f"{N_PIX // 2 + 1} rows from 0 to v_per / 2; |r| {R_EDGES_C.size - 1} ({R_INFO_C['n_sep']} one per separation, from {R_EDGES_C[0]:.4f} to "
      f"{R_INFO_C['r_1']:.4f} cMpc; {R_INFO_C['n_step']} steps of {R_INFO_C['dr']:.4g} cMpc to {R_EDGES_C[-1]:.1f} cMpc; separation vectors per bin "
      f"{R_INFO_C['n_lags'].min()} to {R_INFO_C['n_lags'].max()}, {R_INFO_C['n_merged']} empty bin(s) merged); |k| {K_EDGES_C.size - 1} up to {K_EDGES_C[-1]:.4f} cMpc^-1")
print(f"full coverage of the lattice: |k| <= min(pi / d_perp, pi u / dv) = min({np.pi / D_PERP:.4f}, {np.pi / dv_c:.4f}) cMpc^-1, "
      f"|r| <= L / 2 = {L / 2:.1f} cMpc")

In [ ]:
from skewerjax.summary_stats import GRID_KEYS, ensemble_mean, measure_lattice, skewer_p1d, skewer_xi1d
OVERFLUX_FIELDS = (*LINE_IDS, "total")                          # the overflux of each line and of the total
GRID_C = lattice_grid(N_SK, N_SK, N_PIX, L, D_PERP, **EDGES_C, xi_mu_edges=MU_EDGES_XI, pk_mu_edges=MU_EDGES_P,
                      log=print)                                # the 3D statistics on the lattice, bins EDGES_C and the wedges
P1D_EACH, XI1D_EACH, F3D_EACH = [], [], []
for i in range(n_ens):                                          # every statistic per member of the ensemble, from arrays and geometry only
    K1D, p = skewer_p1d(DELTA[i], DV)                           # P1D (N_lines + 1, N_PIX / 2 + 1): one 1D FFT per skewer [km/s]
    LAGS, x = skewer_xi1d(DELTA[i], DV)                         # xi1D (N_lines + 1, N_PIX / 2 + 1) at every lag j dv
    P1D_EACH.append(dict(p1d=p))
    XI1D_EACH.append(dict(xi1d=x))
    # skewer samples: on the skewers' own lattice, along the line of sight N_PIX pixels, across it the skewer spacing; no deposition, window or interlacing
    F3D_EACH.append({f: measure_lattice(DELTA[i, jf], GRID_C) for jf, f in enumerate(OVERFLUX_FIELDS)})  # P, xi in 3D
P1D = dict(ensemble_mean(P1D_EACH, ("p1d",)), k=K1D)            # the mean over the ensemble and its error, k [s/km]
XI1D = dict(ensemble_mean(XI1D_EACH, ("xi1d",)), lags=LAGS)
FLUX3D = {f: ensemble_mean([r[f] for r in F3D_EACH], GRID_KEYS) for f in OVERFLUX_FIELDS}
AXES3D = GRID_C.axes()                                          # the bins of D1 [cMpc], [cMpc^-1]
print(f"statistics of {ENSEMBLE}: P1D and xi1D of {len(OVERFLUX_FIELDS)} overfluxes along {n_sk} skewers each, the 3D statistics on "
      f"the lattice {GRID_C.shape} (overflux) and {GRID_B.shape} (density)")

### D2 · The $(k_\perp, k_\parallel)$ & $(r_\perp, r_\parallel)$ planes

The white rays: the boundaries of the wedges, of `MU_EDGES_P` in the $k$ maps and of `MU_EDGES_XI` in the $r$ maps
(the same in C1). Every map starts at the origin, $(0, 0)$; in the $k$ maps the grey strip below $k_f/2$ holds the
$k_\perp = 0$ and $k_\parallel = 0$ modes, which no bin of the map takes. What is drawn is set in each call: `show_mu` (the
rays), `show_contours` (the contours) and, here, `show_modes` and `show_pairs`: a small dot in each cell at the mode-weighted
mean $(k_\perp, k_\parallel)$ of its modes, or the pair-weighted mean $(r_\perp, r_\parallel)$ of its separations, where the value of
the cell is computed (`axes["kperp_map"]`, `["kpar_map"]`, `["rperp_map"]`, `["rpar_map"]`): $k_\parallel$ and $r_\parallel$ are one
mode or separation per row, so the dots sit on the row centres; across, the mean over the column moves off its middle.

In [ ]:
from skewerjax.utils.plotting import field_names, fold_periodic
ax3, f3 = AXES3D, FLUX3D                                                 # comoving bins [cMpc], [cMpc^-1]
NAMES = field_names(OVERFLUX_FIELDS)                                     # Lyα, Si III, Si II 1193, Si II 1190, total
tot = dict(f3["total"], axes=ax3)                                        # the total overflux and its bins
fig, ax, _ = plot_overflux_kpar_kperp(tot, kms_per_cMpc=u, kperp_edges=K_EDGES_C, n_sigma=1, show_mu=True, show_contours=False, show_modes=True, kpar_max=1.0*k_nyq_c)
plt.show()
print(f"k map: {ax3['kpar_edges'].size - 1} rows, k_par = 2 pi m / v_per from {2 * np.pi / PERIOD:.2e} to pi / dv = {np.pi / DV:.2e} s/km; "
      f"{ax3['kperp_edges'].size - 1} columns of k_perp (K_EDGES_C), of which {int(np.sum(ax3['n_modes_P_map'].sum(0) == 0))} hold no mode")

In [ ]:
fig, ax, curves = plot_overflux_rpar_rperp(
      tot, kms_per_cMpc=u, rperp_edges=RPERP_EDGES_C, r_bao=R_BAO, delta_v=np.asarray(LINES.delta_v),
      line_ids=LINE_IDS, pairs="all", n_sigma=1, color_scale="symlog", quantity="r2xi",
      show_mu=True, show_contours=True, show_pairs=True,   # the dots: where each cell's xi is computed
)  # every pair, metal-metal included, in its colour: diamond, echo + dashed, echo - dotted; the ring grey dotted
plt.show()
SEP = dv_fold / u                                                        # the line separation in comoving units [cMpc], folded
r_par_max = ax3["rpar"][-1]                                              # the last row: half the period [cMpc]
print(f"map: {ax3['rpar_edges'].size - 1} rows, one per pixel separation j dv (dv = {u * np.diff(ax3['rpar_edges'])[0]:.2f} km/s), "
      f"from 0 to {u * r_par_max:.0f} km/s = v_per / 2; {ax3['rperp_edges'].size - 1} columns of r_perp (RPERP_EDGES_C) up to "
      f"{ax3['rperp_edges'][-1]:.1f} cMpc; u r_BAO = {u * R_BAO:.0f} km/s")
ends = fold_periodic([R_BAO - SEP, R_BAO + SEP], L)
print(f"Delta = Delta v / u = {SEP:.2f} cMpc ({dv_fold:.0f} km/s); the echo branches start on the line of sight at "
      f"|r_BAO - Delta| = {ends[0]:.1f} and r_BAO + Delta = {ends[1]:.1f} cMpc (folded into [0, L / 2]) and meet at "
      f"(r_perp, r_par) = ({R_BAO:g}, {SEP:.1f}) cMpc")
X, Xe = f3["total"]["xi_map"], f3["total"]["xi_map_err"]
ip = int(np.argmin(np.abs(ax3["rpar"] - SEP)))                         # the r_par row of Delta
for q in (0, 1):                                    # r_perp columns: 0 = pairs on the same skewer, 1 = one transverse spacing
    excess = X[ip, q] - 0.5 * (X[ip - 2, q] + X[ip + 2, q])
    print(f"Si III - Lya at r_par = Delta, r_perp bin {q} ({ax3['rperp_edges'][q]:.1f}-{ax3['rperp_edges'][q + 1]:.1f} cMpc): "
          f"xi = {X[ip, q]:.2e}, excess over the rows two pixels away {excess:.2e} = {excess / Xe[ip, q]:.1f} sigma ({ENSEMBLE})")
b_ratio = np.sqrt(f3["SiIII-1206"]["P_iso"][0] / f3["HI-LyA"]["P_iso"][0])  # b_S / b_L from the spectra at the lowest |k|
R2 = np.nan_to_num(ax3["rpar_map"]) ** 2 + np.nan_to_num(ax3["rperp_map"]) ** 2
M, Me = R2 * X, R2 * Xe                                                  # r^2 xi of the map and its error [cMpc^2]

def along(c):                                       # the mean of r^2 xi over the cells a curve crosses, and its error
    i = np.clip(np.searchsorted(ax3["rpar_edges"], np.asarray(c["r_par"]) / u) - 1, 0, M.shape[0] - 1)
    k = np.clip(np.searchsorted(ax3["rperp_edges"], np.asarray(c["r_perp"])) - 1, 0, M.shape[1] - 1)
    cells = np.unique(np.stack([i, k]), axis=1)
    return M[cells[0], cells[1]].mean(), np.sqrt(np.sum(Me[cells[0], cells[1]] ** 2)) / cells.shape[1]

for name, label in (("ring", "r_BAO"), ("echo +", "echo +Delta v (Si III - Lya)"), ("echo -", "echo -Delta v (Si III - Lya)")):
    m, e = along(curves[name])
    print(f"  mean r^2 xi along the {label} curve: {m:+.3f} +- {e:.3f} cMpc^2")
print(f"b_S / b_L = sqrt(P_SiIII / P_Lya) at the lowest |k| = {b_ratio:.3f}: the expected amplitude of the echoes relative to "
      f"the ring")
shared = CHECK["mean_delta"][-1]                                         # <delta_tot> at each pixel, over every skewer of every box
s_glob = drift[-1] / (1 + drift[-1].mean())                              # the pattern a global mean would keep: T_bar(v) / <T_bar> - 1
print(f"pattern shared by every skewer in delta_tot (MEAN = {MEAN!r}, a diagnostic): rms {shared.std():.1e} "
      f"(zero by construction with 'redshift'); with mean='global' it would be the drift of T_bar(v), rms {s_glob.std():.1e}, "
      f"adding about {s_glob.var():.1e} to xi at every r_perp (the k_perp = 0 modes)")

**The echoes of the BAO ring.** On large scales the overflux of each line follows the gas, $\delta_j \simeq b_j\delta_g$,
with a bias $b_j$. Si III is evaluated on the $\mathrm{Ly}\alpha$ axis at $v + \Delta v_{\mathrm{Ly}\alpha\text{–Si III}}$, so it samples the gas a
distance $\Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u$ further along the line of sight:

$$
\delta_{\rm tot}(\mathbf x) \simeq b_L\,\delta_g(\mathbf x) + b_S\,\delta_g\big(\mathbf x + (\Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u)\,\hat{\mathbf y}\big).
$$

Its correlation, with $\xi_g$ that of the gas:

$$
\xi_{\rm tot}(r_\perp, r_\parallel) \simeq \big(b_L^2 + b_S^2\big)\,\xi_g\Big(\sqrt{r_\perp^2 + r_\parallel^2}\Big)
+ b_Lb_S\Big[\xi_g\Big(\sqrt{r_\perp^2 + (r_\parallel - \Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u)^2}\Big) + \xi_g\Big(\sqrt{r_\perp^2 + (r_\parallel + \Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u)^2}\Big)\Big].
$$

The cross terms copy the gas correlation shifted by $\pm\Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u$ along the line of sight: its peak at $r = 0$ becomes the
spot at $(0, \Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u)$ (the diamond), and its BAO ring two rings centred on $(0, \pm\Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u)$. After folding
$r_\parallel \to \lvert r_\parallel\rvert$, both lie on the BAO ring re-centred on the Si III spot:

$$
r_\perp^2 + (r_\parallel - \Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u)^2 = r_{\rm BAO}^2,\qquad
r_\parallel^{\pm}(r_\perp) = \Big\lvert\sqrt{r_{\rm BAO}^2 - r_\perp^2} \pm \Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u\Big\rvert,\quad r_\perp \le r_{\rm BAO}.
$$

- **Ends.** On the line of sight, $r_\perp = 0$, the branches start at $r_\parallel = r_{\rm BAO} \pm \Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u$ (the $-$
  branch at $\lvert r_{\rm BAO} - \Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u\rvert$); they meet at $(r_\perp, r_\parallel) = (r_{\rm BAO}, \Delta v_{\mathrm{Ly}\alpha\text{–Si III}}/u)$.
- **Periodicity.** Line-of-sight separations are periodic with period $v_{\rm per}$ ($L$ in cMpc), so every curve, the
  ring included, is folded with $y \to \min\big(y \bmod v_{\rm per},\ v_{\rm per} - (y \bmod v_{\rm per})\big)$: a branch
  that passes beyond $v_{\rm per}/2$ re-enters closer to 0. Nothing is drawn beyond $r_\perp = L/2$.
- **Amplitude.** The ring carries $(b_L^2 + b_S^2)\,\xi_g$, each echo $b_Lb_S\,\xi_g$: relative to the ring, the echoes
  are of order $b_S/b_L$ (for $b_S \ll b_L$), with $b_S/b_L \simeq \sqrt{P_{\rm SiIII}/P_{{\rm Ly}\alpha}}$ at the
  lowest $\lvert\mathbf k\rvert$ (printed).

The same holds for every pair of lines $(i, j)$ with $\Delta v_{i\text{–}j}$: every pair, metal–metal included, has its
diamond on the line of sight and its echoes on the map (`pairs="all"`; `"reference"` keeps those with $\mathrm{Ly}\alpha$), folded with
the period. Each pair has one colour, named in the legend: the diamond is its spot, the echo $r_\parallel^+$ dashed and $r_\parallel^-$
dotted; the ring is grey and dotted.

### D3 · Power spectrum, correlation function and wedges of the overfluxes

- **$\lvert\mathbf k\rvert \le k_{\rm N} = \min\bigl(\pi N_{\rm pix}/L,\ \pi/(s_x\,dx),\ \pi/(s_z\,dx)\bigr)$**, in the
  bins `K_EDGES_C` (D1), the smallest Nyquist frequency of the lattice ($\pi N_{\rm pix}/L = u\,\pi/dv$ along the line of sight). A
  $\lvert\mathbf k\rvert$ bin averages a shell of modes, and only inside the sphere of radius $k_{\rm N}$ does every
  shell hold modes in every direction. Beyond it, the coarser axes have no modes left, so the shell keeps only the
  directions near the finer axis, and $P(\lvert\mathbf k\rvert)$ would turn into the power along that one direction.
- **$\lvert\mathbf r\rvert \le L/2$**, in the bins `R_EDGES_C` (D1). The lattice is periodic, so is its correlation function,
  $\xi(\mathbf r + L\hat{\mathbf e}_i) = \xi(\mathbf r)$: a separation beyond half the box along an axis is the same as
  its image inside, and adds nothing new. Below the transverse spacing the $\lvert\mathbf r\rvert$ bins hold separations
  along the line of sight only.
- **The shaded limits** come from one helper for the lattice that bins the points (`lattice_limits`, `coverage_limits`;
  C2 alike, `GRID_B.limits()`): $k_{\min} = 2\pi/L$ and $k_{\max} = k_{\rm N}$; $r_{\min}$ the smallest non-zero separation of
  the lattice, $\min(s\,dx, dv/u)$, which the first bin of `R_EDGES_C` holds alone, and $r_{\max} = L/2$. Every point lies
  inside, the first on $r_{\min}$; the wedges below are shaded alike.

**The vertical lines.** Dotted grey, the BAO: its maxima in $k$ and $r_{\rm BAO}$. Solid, in the colour of each pair $(i, j)$, its
separation folded into the period, $\Delta_{ij} = \Delta v_{ij,\rm fold}/u$: in $\xi$ the spot itself; in $P(\lvert\mathbf k\rvert)$ its imprint. Averaged over the
directions, the cross term $\cos(k\mu\Delta_{ij})$ gives $j_0(k\Delta_{ij})$, whose maxima are at $k\Delta_{ij} = 7.725, 14.066, 20.371, \ldots$ (`pair_imprint`; every maximum within the range).

In [ ]:
from skewerjax.summary_stats import bao_height, bao_maxima
K_BAO_MAX = np.pi / dx                                                             # [cMpc^-1]: the maxima up to the Nyquist frequency of the cells
BAO_K = bao_maxima(K_BAO_MAX, window=boxes.grf().bao_window, r_bao=R_BAO)   # the positive maxima of the bump term A_BAO W(k) [cMpc^-1]
fig, axs = plot_overflux_3d(
      f3, ax3, kms_per_cMpc=u, r_bao=R_BAO, bao_k=BAO_K, delta_v=np.asarray(LINES.delta_v), line_ids=LINE_IDS, box=L,
      names=NAMES, k_edges=K_EDGES_C, r_edges=R_EDGES_C, limits=lim, quantity="r2xi", y_scale="linear", k_scale="linear",
      each_label=EACH, n_harmonics=9
)
add_panel_letters([axs["P"], axs["xi"]])
plt.show()
PT = f3["total"]["P_iso"]
print(f"lattice {N_SK} x {N_PIX} x {N_SK} per box, a cube of side L = {L:g} cMpc (v_per = {PERIOD:.0f} km/s along the line of sight)")
print(f"|k| from {ax3['k_edges'][0]:.3f} to k_N = {ax3['k_edges'][-1]:.3f} cMpc^-1 in {ax3['k_edges'].size - 1} bins: Nyquist "
      f"pi N_pix / L = {lim['k_nyq_par']:.3f} cMpc^-1 along the line of sight (pi / dv = {lim['k_nyq_par'] / u:.2e} s/km), "
      f"pi / (s dx) = {lim['k_nyq_perp'][0]:.3f} (x), {lim['k_nyq_perp'][1]:.3f} (z) cMpc^-1 across it")
print(f"|r| from r_min = {lim['r_min']:.3f} cMpc (dv / u, the smallest separation) to r_max = L / 2 = {lim['r_max']:.1f} cMpc in "
      f"{ax3['r_edges'].size - 1} bins (R_EDGES_C), the first centred at {ax3['r_iso'][0]:.3f} cMpc; line-of-sight separations only below the transverse "
      f"spacing {lim['r_perp_min']:.2f} cMpc")
print("P_line / P_tot at the lowest |k|: " + ", ".join(f"{f} {f3[f]['P_iso'][0] / PT[0]:.2f}" for f in LINE_IDS))
r3 = ax3["r_iso"]
d3, d3_th = DENS_STATS["density3d"]["total"], REF_B                    # the density on the same skewers (Part C)
rB = axB["r_iso"]                                                      # its own |r| bins (R_EDGES_B)
bao_rho = bao_height(rB, rB**2 * d3_th["xi_iso"], R_BAO, boxes.sigma_bao)["height"]
print(f"BAO peak of r^2 xi above its sidebands at r_BAO -+ 4 sigma_BAO [cMpc^2], against (P_X / P_density at the lowest |k|) "
      f"x the density's expected {bao_rho:.3f}:")
for f in (*LINE_IDS, "total"):
    bf = bao_height(r3, r3**2 * f3[f]["xi_iso"], R_BAO, boxes.sigma_bao, r3**2 * f3[f]["xi_iso_err"])
    print(f"  {NAMES[f]:>12}: r^2 xi(r_BAO) = {bf['value']:.3f}, height {bf['height']:.3f} +- {bf['height_err']:.3f}; "
          f"scaled density {f3[f]['P_iso'][0] / d3['P_iso'][0] * bao_rho:.3f}")

**Wedges of the total overflux.** The vertical lines are line-of-sight positions only ($r_\perp = 0$, $\mu = 1$), folded into
$[0, L/2]$: $r_{\rm BAO}$, and for every pair the spot $\Delta v_{ij}/u$ and the crossings of its echoes,
$\lvert r_{\rm BAO} \pm \Delta v_{ij}/u\rvert$, in the styles of the maps: $r_{\rm BAO}$ grey and dotted, and in the pair's colour the spot solid,
the echo $+$ dashed and the echo $-$ dotted. In the other wedges the same features sit elsewhere (the map of D2).
Shaded outside $[r_{\min}, r_{\max}]$, the values of the isotropic figure. The points of every wedge sit at the bins of
`R_EDGES_C`; a wedge with no separation in a bin (at small $r$ the few separations of a bin have few directions) is NaN
there: an open marker on the line between two computed bins, nothing at the ends of a wedge.

In [ ]:
from skewerjax.utils.plotting import los_scales, plot_xi_mu_wedges
LOS = los_scales(R_BAO, LINE_IDS, np.asarray(LINES.delta_v) / u, L)   # r_BAO, the spot and the echoes of every pair on the line of sight [cMpc], folded
fig, axs, _ = plot_xi_mu_wedges(tot, r_edges=R_EDGES_C, r_bao=R_BAO, delta_v=np.asarray(LINES.delta_v), line_ids=LINE_IDS, box=L, kms_per_cMpc=u,
                                limits=lim, quantity="xi", y_scale="symlog", each_label=EACH)   # the line-of-sight scales of every pair; shaded as D3
plt.show()
mu_e, r_mu = ax3["mu_edges"], ax3["r_mu"]
jb = np.nanargmin(np.abs(r_mu - R_BAO), axis=1)
print(f"line-of-sight (mu = 1) scales [cMpc], folded into [0, L / 2]: r_BAO {LOS['ring']:.1f}; spot, |r_BAO + Delta|, |r_BAO - Delta| of every pair: " + "; ".join(
      f"{NAMES_L[p['i']]}–{NAMES_L[p['j']]} {p['spot']:.1f}, {p['plus']:.1f}, {p['minus']:.1f}" for p in LOS["pairs"]))
print("r^2 xi of the total at r_BAO [cMpc^2]: " + "; ".join(
      f"{mu_e[w]:.2f} < mu < {mu_e[w + 1]:.2f}: {r_mu[w, j]**2 * f3['total']['xi_mu'][w, j]:+.2f} +- {r_mu[w, j]**2 * f3['total']['xi_mu_err'][w, j]:.2f}"
      for w, j in enumerate(jb)))

**Wedges of $P$ of the total overflux.** The Fourier counterpart of the wedges of $\xi$ above, with its own edges
`MU_EDGES_P` ($\mu = k_\parallel/\lvert\mathbf k\rvert$, $k_\parallel$ along the skewers in cMpc$^{-1}$ through $u$) on
the bins `K_EDGES_C`: the mean over the boxes with its error, each box faint; no expectation (the overflux has
none). The vertical lines:

- dotted grey, the BAO maxima of the bump term (`bao_maxima`), as in the isotropic figure;
- solid in the colour of each pair $(i, j)$, its imprint along the line of sight ($\mu = 1$): the cross term
  multiplies $P$ by $1 + c\cos(k_\parallel\Delta_{ij})$, with maxima at $k = 2\pi n/\Delta_{ij}$,
  $\Delta_{ij} = \Delta v_{ij,\rm fold}/u$ (`pair_imprint`, the lines of P1D in D4). In a wedge of mean $\mu$ the same
  maxima sit at $2\pi n/(\mu\Delta_{ij})$ in $\lvert\mathbf k\rvert$, so the oscillation is sharpest in the wedge
  nearest $\mu = 1$ and washed out across the line of sight.

The points of every wedge sit at the bins of `K_EDGES_C`, at the mode-weighted $\lvert\mathbf k\rvert$ of each bin, as in
D3. A wedge with no mode in a bin (the narrow wedges near $\mu = 1$ at small $k$: the lattice's modes have few
directions there) is NaN: an open marker on the line between two computed bins, nothing at the ends of a wedge.

In [ ]:
from skewerjax.summary_stats import pair_imprint
from skewerjax.utils.plotting import line_pairs, plot_pk_mu_wedges
fig, axs, _ = plot_pk_mu_wedges(tot, k_edges=K_EDGES_C, bao_k=BAO_K, delta_v=np.asarray(LINES.delta_v), line_ids=LINE_IDS, box=L, panels=True,
                                kms_per_cMpc=u, limits=lim, quantity="P", k_scale="linear", each_label=EACH)   # the mu = 1 imprint of every pair
add_panel_letters(axs)
plt.show()
for p in line_pairs(LINE_IDS, np.asarray(LINES.delta_v) / u, L):
    kk = pair_imprint(p["fold"], lim["k_max"], kind="p1d")             # k = 2 pi n / Delta_ij up to k_N
    print(f"{NAMES_L[p['i']]}–{NAMES_L[p['j']]}: Delta = {p['fold']:.2f} cMpc ({u * p['fold']:.0f} km/s), mu = 1 imprint at k = 2 pi n / Delta = "
          f"{np.round(kk, 3)} cMpc^-1 = {np.round(kk / u, 4)} s/km")
mu_p, ke = ax3["pk_mu_edges"], ax3["k_mu_edges"]
jk = int(np.searchsorted(ke, BAO_K[0], side="right") - 1)              # the |k| bin of the first BAO maximum
PW, PWe = f3["total"]["P_mu"][:, jk], f3["total"]["P_mu_err"][:, jk]
print(f"P of the total in the |k| bin {ke[jk]:.4f}-{ke[jk + 1]:.4f} cMpc^-1 of the first BAO maximum (k = {BAO_K[0]:.4f}) [cMpc^3]: " + "; ".join(
      f"{mu_p[w]:.2f} < mu < {mu_p[w + 1]:.2f}: {PW[w]:.4g} +- {PWe[w]:.2g}" for w in range(PW.size)))

**Reading.** The wedge nearest the line of sight ($0.95 < \mu \le 1$) oscillates with the period of the
$\mathrm{Ly}\alpha$–Si III pair, $2\pi/\Delta = 0.20$ cMpc$^{-1}$ ($\Delta = 31.4$ cMpc, printed), its maxima on the solid
lines; the oscillation fades with $\mu$, where the maxima move to $2\pi n/(\mu\Delta)$ and a wedge averages over them,
as in $\xi$, where the spot and the echoes are on the line of sight only. The BAO wiggles, period
$2\pi/r_{\rm BAO} \simeq 0.043$ cMpc$^{-1}$, run through every wedge at the dotted lines. Above
$k \simeq 0.3$ cMpc$^{-1}$, $P$ grows with $\mu$: the overflux is anisotropic, its peculiar velocities and line profiles
acting along the line of sight only. The last wedge drops to $\sim 10^{-15}$ cMpc$^3$ at the smallest $k$: there they hold
only the two modes $k_\perp = 0$, $k_\parallel = \pm k$, and `MEAN = "redshift"` divides every pixel by the mean over every
skewer, so the mean of $\delta$ across the skewers vanishes at every pixel and with it the whole plane $k_\perp = 0$ of $P$
(zero up to rounding; P1D, the sum over $k_\perp$, is not affected). With two boxes the errors are rough (printed at the bin of the first BAO maximum,
where the wedge $\mu > 0.95$ holds few modes).

### D4 · P1D next to a DESI measurement

The DESI DR1 P1D at every redshift of Chaves-Montero et al. (Fig. 1; `utils.reference_data`), one colour per $z$, small points
joined by a thin line. The dotted grey lines are the BAO maxima of the 3D spectrum at $k_\parallel = k/u$, for reference: P1D integrates
over $k_\perp$ and damps them. The solid lines, in the colour of each pair $(i, j)$, are the imprint of its separation folded into the
period: the cross term goes as $\cos(k_\parallel\Delta v_{ij})$, with maxima at $k_\parallel = 2\pi n/\Delta v_{ij,\rm fold}$ (`pair_imprint`; the first `N_HARMONICS`
of each pair).

**From 3D to 1D.** With $k_\parallel$, $r_\parallel$ along the line of sight and $\mathbf k_\perp$ across it:

$$
P_{\rm 1D}(k_\parallel) = \int\frac{d^2k_\perp}{(2\pi)^2}\,P_{\rm 3D}(k_\parallel, \mathbf k_\perp)
= \frac{1}{2\pi}\int_{\lvert k_\parallel\rvert}^\infty k\,P(k)\,dk\ \ \text{(isotropic)},\qquad
\xi_{\rm 1D}(r_\parallel) = \xi_{\rm 3D}(r_\perp = 0, r_\parallel) = \int\frac{dk_\parallel}{2\pi}\,e^{ik_\parallel r_\parallel}\,P_{\rm 1D}(k_\parallel).
$$

On the lattice, $P_{\rm 1D}(k_\parallel) = L^{-2}\sum_{\mathbf k_\perp}P(k_\parallel, \mathbf k_\perp)$, the form of
`summary_stats.density._lattice_p1d` (the expectation of the density along the skewers; no P1D expectation is drawn in this
notebook); velocity units through $u$: $k_\parallel\,[\mathrm{s/km}] = k_\parallel\,[\mathrm{cMpc^{-1}}]/u$,
$P_{\rm 1D}\,[\mathrm{km/s}] = u\,P_{\rm 1D}\,[\mathrm{cMpc}]$.

**How they are measured.** P1D and $\xi_{\rm 1D}$ are 1D FFTs of $\delta$ along each skewer, averaged over the
skewers, not derived from the 3D statistics (`summary_stats/lattice.py`: `skewer_p1d`, `skewer_xi1d`; D1).
P1D is shown on its native modes $k_m = 2\pi m/v_{\rm per}$, $m = 1, \dots, N_{\rm pix}/2$, one point per
mode (no rebinning: the mean over the boxes with the error of the mean, every box faint), from
$k_{\min} = 2\pi/v_{\rm per}$ to $k_{\max} = \pi/dv$; the resolution is set by $dv$ and $v_{\rm per}$. $\xi_{\rm 1D}$ is shown
at every lag $r_\parallel = j\,dv$ from 0 to $v_{\rm per}/2$.

In [ ]:
N_HARMONICS = 13                                                     # the first maxima of every pair's imprint on the P1D figures (all of them would be 100-200 lines)
from skewerjax.utils.plotting import plot_p1d_reference
from skewerjax.utils.reference_data import P1D_DESI_DR1, nearest_reference
P1D_REF = nearest_reference(P1D_DESI_DR1, Z)                               # the reference nearest to the box's z, for the print (every z is drawn alike)
K_LIMITS_1D = (2 * np.pi / PERIOD, np.pi / DV)                                   # k_min, k_max of the line of sight [s/km]
BAO_K_1D = bao_maxima(P1D_REF["k_max_figure"] * u, window=boxes.grf().bao_window, r_bao=R_BAO, kms_per_cMpc=u)  # k_par = k / u of the 3D maxima [s/km]
fig, ax = plot_p1d_reference(
      P1D, P1D_DESI_DR1, names=NAMES, k_max=P1D_REF["k_max_figure"],                     # the native modes (no k_edges)
      k_limits=K_LIMITS_1D, kms_per_cMpc=u, each_label=EACH, bao_k=None,
      delta_v=np.asarray(LINES.delta_v), line_ids=LINE_IDS, n_harmonics=N_HARMONICS
)
plt.show()
kc = P1D["k"]
toy = np.interp(P1D_REF["k"], kc, kc * P1D["p1d"][0] / np.pi)
print(f"toy z = {Z:g}, the {len(P1D_DESI_DR1)} reference redshifts {min(P1D_DESI_DR1):g} to {max(P1D_DESI_DR1):g}, the nearest z = {P1D_REF['z']:g}; reference / toy (Lya) over its k range: "
      f"median {np.median(P1D_REF['kP_pi'] / toy):.0f}")
print("line separations: periods 2 pi / Delta v [s/km]: " + ", ".join(f"{NAMES_L[p['i']]}–{NAMES_L[p['j']]} {2 * np.pi / p['dv']:.4f}" for p in PAIRS) + "; "
      f"Nyquist frequency of the cells along the skewers pi / (u dx) = {np.pi / (u * dx):.4f} s/km")
print(f"P1D on its {kc.size - 1} native modes k_m = 2 pi m / v_per (no rebinning), from k_min = 2 pi / v_per = {kc[1]:.2e} "
      f"to k_max = pi / dv = {kc[-1]:.2e} s/km")

## Part E — The decomposition by order

P1D and $\xi_{\rm 1D}$ of the total overflux (Part D) split **exactly** into the contributions of the pairs of subset
terms and their orders: order by order (E1) and term by term (E2). The decomposition is computed only here, by the
explicit call `decompose_1d` of E1 (opt-in), one FFT per skewer. The same decomposition of the 3D statistics on the
lattice (every map, wedge and isotropic bin of Part D, with P1D and $\xi_{\rm 1D}$ as its reductions) is
`summary_stats.decompose` (`docs/decomposition.md`), drawn in the $\mu$ wedges in Part F.

### E1 · The decomposition by order

**The terms.** `DeltaFieldBuilder` also forms, per skewer, the products $\delta_A = \prod_{j \in A}\delta_j$ of every
subset $A$ of lines. With $\mathcal C$, the means of the products over the set of pixels of the mean transmission, $\mathcal C = \langle\delta_1\delta_2\rangle$
for two lines (plus the higher orders for more), the total overflux decomposes **exactly** into them (review notes, Sec.
Overflux, with the mean of the ensemble):

$$
\delta_{\rm tot} = \frac{\sum_j\delta_j + \sum_{j_1 < j_2}\delta_{j_1}\delta_{j_2} + \cdots - \mathcal C}{1 + \mathcal C},\qquad
\mathcal C = \sum_{\lvert A\rvert \ge 2}\langle\delta_A\rangle = \frac{\bar T}{\prod_j \bar T_j} - 1.
$$

It follows from $T = \prod_j \bar T_j(1 + \delta_j)$ and its mean over the same set,
$\bar T = \prod_j \bar T_j\,\bigl(1 + \sum_j\langle\delta_j\rangle + \mathcal C\bigr)$. The order-1 term of the
denominator, $\sum_j\langle\delta_j\rangle$, vanishes because every $\bar T_j$ and $\bar T$ are
means over the same set of pixels (every mode of `mean`). With the default `"redshift"`, $\mathcal C(v)$ is
$\langle\delta_1\delta_2\rangle$ at each pixel over the ensemble (one number with `"global"`, one per box with `"box"`,
one per skewer with `"skewer"`; Part B prints the largest difference, at machine precision), and the sum of the terms
reproduces $\delta_{\rm tot}$ at every pixel of every skewer to machine precision (printed).
`OverfluxStatistics1D` gives P1D and $\xi_{\rm 1D}$ of the total, of every pair of terms $(A, B)$ and of every order
$\lvert A\rvert + \lvert B\rvert$: order $n$ collects the terms with $\lvert A\rvert + \lvert B\rvert = n$, order 0 is the
normalisation term $\mathcal C$, and the orders add up to the total to machine precision (printed).

**The terms of two lines** (the example; four lines add every subset, orders up to 8). With two lines, at each pixel,

$$
T = \bar T_1 \bar T_2\, (1 + \delta_1)(1 + \delta_2),\qquad
\delta = \frac{\delta_1 + \delta_2 + \delta_1\delta_2 - \mathcal C}{1 + \mathcal C},\qquad \mathcal C = \langle \delta_1 \delta_2 \rangle.
$$

Every term of $\xi_{\rm 1D}$ (real space) or P1D (Fourier space) is a pair of subsets $A, B$ of $\{1, 2\}$ (or the
normalisation term), of order $\lvert A\rvert + \lvert B\rvert$:

| $A \backslash B$ | $\mathcal{C}$ (order 0) | 1 (order 1) | 2 (order 1) | 12 (order 2) |
|---|---|---|---|---|
| $\mathcal{C}$ | $+\mathcal{C}\mathcal{C}$ | 0 | 0 | $-\mathcal{C}\mathcal{C}$ |
| 1 | 0 | $\xi_{11}$, $P_{11}$ | $\xi_{21}$, $P_{21}$ | $\xi_{(12)1}$, $P_{(12)1}$ |
| 2 | 0 | $\xi_{12}$, $P_{12}$ | $\xi_{22}$, $P_{22}$ | $\xi_{(12)2}$, $P_{(12)2}$ |
| 12 | $-\mathcal{C}\mathcal{C}$ | $\xi_{1(12)}$, $P_{1(12)}$ | $\xi_{2(12)}$, $P_{2(12)}$ | $\xi_{(12)(12)}$, $P_{(12)(12)}$ |

The zeros are exact after the ensemble average, because $\langle\delta_j\rangle = 0$ over the set of the mean
(here at each pixel): a pair $(\mathcal C, j)$ is $-\mathcal C$ times the mean of $\delta_j$ along the skewer in
$\xi_{\rm 1D}$, and lives at $k = 0$ alone in P1D. The ensemble average gives

$$
\xi = \frac{-\mathcal{C}^2 + \xi_{11} + \xi_{22} + \xi_{12,{\rm sym}} + \xi_{1(12),{\rm sym}} + \xi_{2(12),{\rm sym}} + \xi_{(12)(12)}}{(1 + \mathcal C)^2},
$$

$$
P(k) = \frac{P_{11} + P_{22} + P_{12,{\rm sym}} + P_{1(12),{\rm sym}} + P_{2(12),{\rm sym}} + P_{(12)(12)} - 2\pi\,\mathcal C^2\, \delta_D(k)}{(1 + \mathcal C)^2},
$$

where "sym" adds the pair and its transpose: 1/2 with 2/1, 1/12 with 12/1, and 2/12 with 12/2 (written for a constant
$\mathcal C$). With the default `"redshift"`, $\mathcal C(v)$ varies slowly along the skewer, with the mean
transmission, and the code weights every term at its own pixel by $1/(1 + \mathcal C(v))$, which keeps the
decomposition exact.

For a second line that is a shifted copy of the first, $\delta_2(v) \propto \delta_1(v - \Delta v_{\mathrm{Ly}\alpha\text{–Si III}})$, the cross term is
$P_{12,{\rm sym}} \propto 2\cos(k\Delta v_{\mathrm{Ly}\alpha\text{–Si III}})\, P_{11}$. It oscillates with period $2\pi/\Delta v_{\mathrm{Ly}\alpha\text{–Si III}}$ in $k$ (D4, and
the cross terms of E2): the Fourier image of the $\xi_{\rm 1D}$ peak at $\Delta v_{\mathrm{Ly}\alpha\text{–Si III}}$.

**By order.** Order 2 holds the Gaussian-like products and dominates; orders 3 to $2N$ ($N$ = 4 lines) are the non-linear
corrections of the products of the lines. The line separations $\Delta v_{i\text{–}j}$, folded into the period (solid, each pair in its colour and named by its line; B1 prints them), are where
the cross terms put a peak in $\xi_{\rm 1D}$; in P1D they oscillate as $\cos(k\Delta v_{\mathrm{Ly}\alpha\text{–Si III}})$, with period $2\pi/\Delta v_{\mathrm{Ly}\alpha\text{–Si III}}$
in $k$ (printed below in units of the mode spacing), and the solid lines are their maxima, $k = 2\pi n/\Delta v_{ij,\rm fold}$. The top axes give $k_\parallel$ and the separation in comoving units
(factor $u$).

**The decomposition by order, optional.** `decompose_1d` splits P1D and $\xi_{\rm 1D}$ of the total into the
pairs of subset terms and their orders. It needs the products of the lines at every pixel, so it costs more than the
plain statistics of D1 (`skewer_p1d`, `skewer_xi1d`, `measure_lattice`) and is needed only here: it is called explicitly, not by default.

In [ ]:
from skewerjax.summary_stats import decompose_1d, ensemble_1d
from skewerjax.utils.plotting import plot_p1d_orders, plot_xi1d_orders
FLUX_ORDERS = ensemble_1d([decompose_1d(TAU[i], DV, T_BAR[i], mean=MEAN) for i in range(n_ens)])  # opt-in
dv_lines = np.asarray(LINES.delta_v)                                             # the offsets of every line: every pair's separation
fig, axs = plot_p1d_orders(FLUX_ORDERS, k_limits=K_LIMITS_1D, kms_per_cMpc=u, each_label=EACH, k_scale="linear", delta_v_list=dv_lines, line_ids=LINE_IDS, n_harmonics=N_HARMONICS)
add_panel_letters(axs)
plt.show()
fig, axs = plot_xi1d_orders(FLUX_ORDERS, delta_v_list=dv_lines, line_ids=LINE_IDS, kms_per_cMpc=u, each_label=EACH, y_scale="symlog")  # every lag, 0 .. v_per / 2 (quantity="r2xi", y_scale="symlog": other views)
add_panel_letters(axs)
plt.show()
e = FLUX_ORDERS
print(f"max |sum of the orders - total| / max |total|: P {np.abs(e['P_order'].sum(0) - e['P_total']).max() / np.abs(e['P_total']).max():.1e}, "
      f"xi {np.abs(e['xi_order'].sum(0) - e['xi_total']).max() / np.abs(e['xi_total']).max():.1e}; at every pixel: "
      f"max |[sum_A delta_A - C] / (1 + C) - delta_tot| = {CHECK['decomposition_error']:.1e}; order 1 / max |total|: "
      f"P {np.abs(e['P_order'][1]).max() / np.abs(e['P_total']).max():.1e}")
xi_c, xi_d = e["xi_total"][N_PIX // 2:], XI1D["xi1d"][-1][: N_PIX - N_PIX // 2]  # the lags 0 dv .. (N_PIX - N_PIX // 2 - 1) dv, odd N_PIX too (centred lags: 0 at N_PIX // 2)
print(f"the total of the decomposition against the per-skewer statistics of D1: P1D {np.abs(e['P_total'] - P1D['p1d'][-1]).max() / P1D['p1d'][-1].max():.1e}, "
      f"xi1D {np.abs(xi_c - xi_d).max() / np.abs(xi_d).max():.1e} (relative)")
print("orders: " + ", ".join(str(n) for n in range(e["P_order"].shape[0])) + f" ({len(LINE_IDS)} lines: 0 .. 2 N_lines); cos(k Delta v) of Si III - "
      f"Lya: period 2 pi / Delta v = {2 * np.pi / abs(dv_12):.2e} s/km = {PERIOD / abs(dv_12):.2f} mode spacings (2 pi / period)")

### E2 · Each order, term by term

The same decomposition one level down, one row per order. Every term $(A, B)$ of an order is drawn with its transpose
$(B, A)$ as A–B (their sum, which is real), on the order's total (the thick line). The lines are numbered 1 to 4 (`LINE_IDS`, written in the first legend of the P1D figure). Order 2 holds the products $j$–$j$, the cross
terms $i$–$j$ and the normalisation cross terms $\mathcal{C}$–$ij$; orders 3 to 8 the products with the subsets of two or more
lines. With four lines an order holds up to 28 terms: the six largest are drawn (`max_terms`; `terms=` selects others), the
rest summed (grey dashed). The
order-0 term $\mathcal{C}$–$\mathcal{C}$ is not shown: it is nearly constant in $\xi_{\rm 1D}$, and lives at the lowest
$k$ in P1D.

In [ ]:
from skewerjax.utils.plotting import plot_p1d_terms, plot_xi1d_terms
fig, axs = plot_p1d_terms(FLUX_ORDERS, k_limits=K_LIMITS_1D, kms_per_cMpc=u, line_ids=LINE_IDS, k_scale="linear", delta_v_list=dv_lines, n_harmonics=N_HARMONICS)  # the largest terms of each order
add_panel_letters(axs)
plt.show()
fig, axs = plot_xi1d_terms(FLUX_ORDERS, delta_v_list=dv_lines, kms_per_cMpc=u, line_ids=LINE_IDS, y_scale="symlog")  # the separations, folded: solid, in the pairs' colours
add_panel_letters(axs)
plt.show()

## Part F — The wedges by order

The 3D statistics of Part D reduced to the $\mu$ wedges of D3, $P(k, \mu)$ and $\xi(r, \mu)$ of the total overflux, split
**exactly** into the orders and terms of Part E: order by order (F1) and term by term (F2). The decomposition is the 3D one
of `summary_stats.decompose` (`docs/decomposition.md`), reduced to the wedges only, by the explicit call of F1 (opt-in).

### F1 · The wedges by order

**A wedge** is the mean of a 3D statistic over the modes with $\lvert\mathbf k\rvert$ in a bin and $\mu = k_\parallel/\lvert\mathbf k\rvert$
in an interval (for $\xi$, the separations with $\lvert\mathbf r\rvert$ in a bin and $\mu = r_\parallel/r$), half-open, the last
interval closed at $\mu = 1$. Here they are those of D3: the 16 wedges of `MU_EDGES_P` for $P$ and the 4 of `MU_EDGES_XI`
for $\xi$, on the bins `K_EDGES_C`, `R_EDGES_C`, with the limits of D3 (`GRID_C`, `lim`), $k_\parallel$ and $r_\parallel$ in
comoving units through $u$ (Part D).

**From 3D to wedges.** With the fields of E1, $F_\emptyset = -\mathcal C/(1+\mathcal C)$ and $F_A = \delta_A/(1+\mathcal C)$,
the total is $\delta_{\rm tot} = \sum_A F_A$ at every pixel. The lattice spectrum is bilinear,
$P_{AB}(\mathbf k) = (dV/N)\,F_A(\mathbf k)\,F_B^*(\mathbf k)$, with $\xi_{AB}$ its inverse transform, and a wedge is a mean over
its modes, a linear operation. So, in every wedge and every bin, for each box,

$$
P(k, \mu) = \sum_{n=0}^{2N} P^{(n)}(k, \mu),\qquad P^{(n)}(k, \mu) = \sum_{\lvert A\rvert + \lvert B\rvert = n} P_{AB}(k, \mu),
$$

and the same for $\xi(r, \mu)$ ($N$ = 2 lines: orders 0 to 4). The terms are the symmetric pairs
$P_{AB} + P_{BA} = 2\,{\rm Re}\,P_{AB}$: the antisymmetric parts, odd in $k_\parallel$, cancel in a wedge, which folds
$\pm k_\parallel$ into $\mu \ge 0$. $\mathcal C$ is the same for every skewer, so the $\mathcal C$ terms (orders 0 and 1, and
$\mathcal C$–12 in order 2) live on the plane $k_\perp = 0$: in $P$ only in the last wedge, in $\xi$ as a function of $r_\parallel$
alone. The checks (printed): $\max\lvert\sum_n P^{(n)} - P\rvert$ in each wedge, relative to the largest value of that wedge
(`wedge_sum_rule`), and the total against the wedges of D3.

In [ ]:
from skewerjax.summary_stats import decompose, ensemble_decomposition
from skewerjax.utils.plotting import plot_pk_mu_orders
WEDGES = ensemble_decomposition([decompose(TAU[i], DV, T_BAR[i], grid=GRID_C, mean=MEAN, group="both", reductions=("P_mu", "xi_mu"))
                                 for i in range(n_ens)])                     # opt-in: P and xi in the wedges of D3, by order and by pair
fig, axs, CHECK_P = plot_pk_mu_orders(WEDGES, axes=AXES3D, k_edges=K_EDGES_C, limits=lim, delta_v=dv_lines, line_ids=LINE_IDS, box=L, kms_per_cMpc=u)
add_panel_letters(axs)
plt.show()
mu_p, O, PT3 = AXES3D["pk_mu_edges"], WEDGES["P_mu_order"], FLUX3D["total"]["P_mu"]
print("P: max |sum of the orders - total| / max(|total|, |orders|) per wedge (wedge_sum_rule): " + ", ".join(
      f"{mu_p[w]:.3f}-{mu_p[w + 1]:.3f} {v:.1e}" for w, v in enumerate(CHECK_P["sum_rule"])))
print(f"P: the total of the decomposition against the wedges of D3: {np.nanmax(np.abs(WEDGES['P_mu_total'] - PT3)) / np.nanmax(np.abs(PT3)):.1e} "
      "(relative); max |order| / max |total| over the wedges: " + ", ".join(
      f"{n} {np.nanmax(np.abs(O[n])) / np.nanmax(np.abs(WEDGES['P_mu_total'])):.1e}" for n in range(O.shape[0])))

**Reading.** In every wedge order 2 is the total to the eye (the thick grey line under the blue points): order 3, the
products of three overfluxes, changes sign from bin to bin, below 1 % of the largest total (printed); order 4 is a
nearly flat positive floor, about 0.1 % of the total in each bin. Orders 0 and 1 vanish except in the last wedge ($0.995 < \mu \le 1$) below
$k \simeq 0.2$ cMpc$^{-1}$, whose cells hold only $k_\perp = 0$ modes: there `MEAN = "redshift"` makes the total vanish (D3),
and orders 0, 2 and 4 cancel exactly ($+3$, $-6$ and $+3 \times 10^{-4}$ cMpc$^3$ at $k \simeq 0.1$ cMpc$^{-1}$). The residuals
are rounding, $\le 3.5 \times 10^{-7}$ (printed). The solid lines, the $\mu = 1$ imprint of the pair (D3): the total
oscillates with them only in the wedges nearest $\mu = 1$; F2 shows why.

In [ ]:
from skewerjax.utils.plotting import plot_xi_mu_orders
fig, axs, CHECK_XI = plot_xi_mu_orders(WEDGES, axes=AXES3D, r_edges=R_EDGES_C, limits=lim, delta_v=dv_lines, line_ids=LINE_IDS, box=L, kms_per_cMpc=u)
add_panel_letters(axs)
plt.show()
mu_x, O, XT3 = AXES3D["mu_edges"], WEDGES["xi_mu_order"], FLUX3D["total"]["xi_mu"]
print("xi: max |sum of the orders - total| / max(|total|, |orders|) per wedge (wedge_sum_rule): " + ", ".join(
      f"{mu_x[w]:.2f}-{mu_x[w + 1]:.2f} {v:.1e}" for w, v in enumerate(CHECK_XI["sum_rule"])))
print(f"xi: the total of the decomposition against the wedges of D3: {np.nanmax(np.abs(WEDGES['xi_mu_total'] - XT3)) / np.nanmax(np.abs(XT3)):.1e} "
      "(relative); max |order| / max |total| over the wedges: " + ", ".join(
      f"{n} {np.nanmax(np.abs(O[n])) / np.nanmax(np.abs(WEDGES['xi_mu_total'])):.1e}" for n in range(O.shape[0])))

**Reading.** Order 2 is again the total in every wedge; order 3 reaches 0.4 % of the largest total (printed; at the smallest
separations, whose bins hold few lags) and order 4 $10^{-4}$; the $\mathcal C$ terms (orders 0 and 1) stay below $10^{-5}$ of
it. The wedge nearest the line of sight ($0.95 < \mu \le 1$) has the spot of the pair at $r = \Delta = 31.4$ cMpc (the solid
line), as in D3; the other wedges do not. Residuals $\le 3.4 \times 10^{-7}$.

### F2 · Each order, term by term

The same decomposition one level down, as in E2: one block of panels per order, one panel per wedge, every term $(A, B)$
with its transpose as A–B (their sum, real) on the order's total (thick), in the colours of E2. For $P$ order 2 only (16
wedges; `orders=` draws the others), for $\xi$ every order. With two lines order 2 holds the autos 1–1 and 2–2, the cross
1–2 and the normalisation term $\mathcal C$–12.

**The cross term is not isotropic.** For a second line that follows the first shifted by $\Delta$ along the line of sight,
$P_{12,{\rm sym}} \propto 2\cos(k_\parallel\Delta)$ (E1), and $k_\parallel = \mu k$: in a wedge of mean $\mu$ it oscillates in
$\lvert\mathbf k\rvert$ with the period $2\pi/(\mu\Delta)$, its zeros at $k = (2n + 1)\pi/(2\mu\Delta)$ (printed: the first sign
change against the first zero above the wedge's first bin with modes). In
configuration space it is the spot at $r_\parallel = \Delta$, $r_\perp = 0$, which a wedge of $\mu$ passes at the distance
$\Delta\sqrt{1 - \mu^2}$: only the wedges near $\mu = 1$ reach it.

In [ ]:
from skewerjax.utils.plotting import plot_pk_mu_terms
fig, axs, _ = plot_pk_mu_terms(WEDGES, orders=(2,), axes=AXES3D, k_edges=K_EDGES_C, limits=lim, delta_v=dv_lines, line_ids=LINE_IDS, box=L,
                               kms_per_cMpc=u)   # order 2, term by term, in every wedge of P
add_panel_letters(axs)
plt.show()
PAIRS_F = [tuple(p) for p in WEDGES["pairs"]]                                  # the pairs of terms A-B (symmetric sums)
P12, P11, P22 = (WEDGES["P_mu_pair"][PAIRS_F.index(p)] for p in (("1", "2"), ("1", "1"), ("2", "2")))
D12 = line_pairs(LINE_IDS, dv_lines / u, L)[0]["fold"]                         # the separation of lines 1 and 2 [cMpc], folded
def first_sign_change(y, k):                                                   # the first |k| bin where y changes sign (NaN: none)
    ok = np.isfinite(y)
    ch = np.flatnonzero(np.diff(np.sign(y[ok])))
    return k[ok][ch[0]] if ch.size else np.nan
def first_zero(w):                                                             # the first zero (2n + 1) pi / (2 mu_c Delta) of cos(mu_c k Delta) above the first bin of wedge w with modes
    z0 = np.pi / ((mu_p[w] + mu_p[w + 1]) * D12)                               # mu_c: the middle of the wedge
    k0 = AXES3D["k_mu"][w][np.isfinite(P12[w])][0]
    return z0 * (2 * max(np.ceil((k0 / z0 - 1) / 2), 0) + 1)
print(f"cross term 1-2 (Delta = {D12:.2f} cMpc): first sign change in |k| [cMpc^-1], measured (the first zero of cos(mu_c k Delta) above the wedge's first bin with modes): " + ", ".join(
      f"{mu_p[w]:.2f}-{mu_p[w + 1]:.2f} {first_sign_change(P12[w], AXES3D['k_mu'][w]):.3f} ({first_zero(w):.3f})" for w in range(mu_p.size - 1)))
print("per wedge, max |1-2| / max |1-1|: " + ", ".join(f"{np.nanmax(np.abs(P12[w])) / np.nanmax(np.abs(P11[w])):.2g}" for w in range(mu_p.size - 1))
      + "; median 2-2 / 1-1: " + ", ".join(f"{np.nanmedian(P22[w] / P11[w]):.2g}" for w in range(mu_p.size - 1)))

**Reading.** The autos are smooth and positive in every wedge, Si III's (2–2) about five times $\mathrm{Ly}\alpha$'s (1–1)
here (printed), as in E2. The cross 1–2 oscillates with a period that shrinks as $\mu$ grows: positive up to $k_{\max}$
across the line of sight ($\mu < 0.1$, where $\pi/(2\mu\Delta) \simeq 1.0$ cMpc$^{-1}$ lies beyond the range), then its
first sign change follows the zeros of $\cos(\mu k\Delta)$ (printed: 0.28 against 0.34 at $0.10 < \mu < 0.20$, 0.21 against
0.21 at $0.20 < \mu < 0.29$; in the wedges near $\mu = 1$, whose lowest bins hold no mode, the second zero,
$3\pi/(2\mu\Delta) \simeq 0.15$ cMpc$^{-1}$), and near $\mu = 1$ it takes the period of the solid lines, $2\pi/\Delta = 0.20$ cMpc$^{-1}$. The cross term alone
makes the anisotropic oscillation of the total (F1). $\mathcal C$–12 is zero except in the $k_\perp = 0$ cells of the last
wedge, where it cancels the autos and the cross (F1).

In [ ]:
from skewerjax.utils.plotting import plot_xi_mu_terms
fig, axs, _ = plot_xi_mu_terms(WEDGES, axes=AXES3D, r_edges=R_EDGES_C, limits=lim, delta_v=dv_lines, line_ids=LINE_IDS, box=L, kms_per_cMpc=u)
add_panel_letters(axs)
plt.show()
X12, X11, X22 = (WEDGES["xi_mu_pair"][PAIRS_F.index(p)] for p in (("1", "2"), ("1", "1"), ("2", "2")))
jd = np.nanargmin(np.abs(AXES3D["r_mu"] - D12), axis=1)                        # the |r| bin of the spot in every wedge
print(f"xi in the |r| bin of the spot (r ~ Delta = {D12:.1f} cMpc), the cross 1-2 (the autos 1-1, 2-2): " + "; ".join(
      f"{mu_x[w]:.2f}-{mu_x[w + 1]:.2f} {X12[w, j]:+.2e} ({X11[w, j]:+.2e}, {X22[w, j]:+.2e})" for w, j in enumerate(jd)))

**Reading.** In order 2 the autos carry the correlation across the line of sight. In the wedge nearest it the cross 1–2
rises to $7 \times 10^{-5}$ at $r \simeq \Delta$, about 100 times the autos there (printed): the spot of D3 and F1 is the cross term
alone. At $0.8 < \mu < 0.95$ a weaker bump remains ($6 \times 10^{-6}$); across the line of sight it is gone. Orders 3 (1–12,
2–12) and 4 (12–12) are noise-like corrections, below 2 % of the total.